> 在线讲义：[第 16 章 · AMP：不逐帧跟踪，只要求「动作像人」](https://wikieden.github.io/robot-motion-rl/lessons/16-amp.html) · 本书《机器人运控与强化学习》

# 第 16 章 · AMP：不逐帧跟踪，只要求「动作像人」

配套讲义：仓库里 `robot-motion-rl/notes/lessons/16-amp.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

（D 的值和任务奖励是编的，公式与权重取自 AMP 论文）

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第二步 判别器损失 | (D−1)² + (D+1)² | `lsgan_disc_loss(...)` | 0.05、1.95 |
| 第三步 风格奖励 | 式 7 | `style_reward(d)` | 1.00、0.75、0.00、0.75、0.00 |
| 第四步 总奖励 | 式 4 | `total_reward(0.8, 0.75)` | 0.775；另一例 0.6 |

公式和权重来自 AMP 论文式 4、式 7 与表（amp-2104.02180.txt L277、L302、L517）。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

**函数 `style_reward`**：式 7：判别器输出 D 越接近 1（像参考动作）奖励越高，最低截到 0。

In [ ]:
def style_reward(d):
    """式 7：判别器输出 D 越接近 1（像参考动作）奖励越高，最低截到 0。"""
    return max(0.0, 1.0 - 0.25 * (d - 1.0) ** 2)

**函数 `total_reward`**：式 4：r = w_G·r_G + w_S·r_S，论文里两个权重都是 0.5。

In [ ]:
def total_reward(r_task, r_style, w_task=0.5, w_style=0.5):
    """式 4：r = w_G·r_G + w_S·r_S，论文里两个权重都是 0.5。"""
    return w_task * r_task + w_style * r_style

**函数 `lsgan_disc_loss`**：最小二乘判别器损失：参考动作的 D 往 1 推，策略动作的 D 往 −1 推。

In [ ]:
def lsgan_disc_loss(d_real, d_fake):
    """最小二乘判别器损失：参考动作的 D 往 1 推，策略动作的 D 往 −1 推。"""
    return float(np.mean((np.asarray(d_real) - 1) ** 2) + np.mean((np.asarray(d_fake) + 1) ** 2))

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第三步 风格奖励")
for d in (1.0, 0.0, -1.0, 2.0, 3.0):
    print(f"D = {d:+.0f} → r_S = {style_reward(d):.2f}")

In [ ]:
print("\n第三步 判别器损失")
print("参考 D = [0.9, 1.1]，策略 D = [−0.8, −1.2] →", round(lsgan_disc_loss([0.9, 1.1], [-0.8, -1.2]), 3))
print("参考 D = [0.2, 0.4]，策略 D = [0.3, 0.1] →", round(lsgan_disc_loss([0.2, 0.4], [0.3, 0.1]), 3))

In [ ]:
print("\n第四步 和任务奖励相加")
print("任务 0.8、D = 0（风格 0.75）→ r =", total_reward(0.8, style_reward(0.0)))
print("任务 0.2、D = 1（风格 1.00）→ r =", total_reward(0.2, style_reward(1.0)))

## 3. 代码里才看得到的

- 判别器的输入是状态转移 $(s_t, s_{t+1})$，不含动作 a（原文 L220：$r^S(s_t, s_{t+1})$）；所以动作库只需要动捕的姿态序列，不需要知道人「出了多大力」。

## 4. 自己改一改

- 把式 7 里的 0.25 改成 1，看 D = 0 时风格奖励变成多少，「截到 0」的范围变宽还是变窄。
- 把权重改成 w_G = 0.9、w_S = 0.1，重算第四步两个例子，看哪个情况更占便宜。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 跟踪逐帧照视频，AMP 只求看着像；
> 裁判把真的推向一、假的推向负一，分不清时就像了；
> 一减四分之一 (D−1)²，截到零，D 越近一分越高；
> 任务风格各一半，互相牵制不跑偏。